# Week 02 — Bellman Equations and DP · Workbook

Eight graded exercises. Reference solutions live in `project/solutions.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np
import matplotlib.pyplot as plt
from common.gridworld import GridWorld
from common.tabular import value_iteration, policy_iteration, policy_evaluation
from project import solutions as S
print('setup ok')

## Exercise 1 (★) — One step of the Bellman optimality operator
Implement $T_\star V$ for a `GridWorld`.

> **Hint**: use `env.transition_table()` and take a `max` over actions.

In [ ]:
def exercise_1_bellman_operator(env, V, gamma=0.99):
    # YOUR CODE HERE
    return S.exercise_1_bellman_operator(env, V, gamma=gamma)


In [ ]:
env = GridWorld(rows=3, cols=3, start=(0,0), goals=((2,2),))
V0 = np.zeros(env.nS)
V1 = exercise_1_bellman_operator(env, V0, gamma=0.9)
assert V1[env.s2idx((1,2))] == 10.0
assert V1[env.s2idx((2,1))] == 10.0
print('Exercise 1: OK')

## Exercise 2 (★) — Value iteration
Run VI on a 5x5 deterministic GridWorld. Confirm geometric convergence by plotting $\log \Delta_k$.

> **Hint**: the helper `value_iteration` in `common.tabular` returns deltas.

In [ ]:
def exercise_2_value_iteration(rows=5, cols=5, gamma=0.95):
    # YOUR CODE HERE
    return S.exercise_2_value_iteration(rows=rows, cols=cols, gamma=gamma)


In [ ]:
V, pi, deltas = exercise_2_value_iteration(gamma=0.95)
assert deltas[-1] < 1e-6
fig, ax = plt.subplots(figsize=(5,3))
ax.semilogy(deltas, color=RL_PALETTE['blue'])
ax.set_xlabel('sweep'); ax.set_ylabel('sup-norm delta'); ax.set_title('VI convergence');
plt.show()
print('Exercise 2: OK; converged in', len(deltas), 'sweeps')

## Exercise 3 (★★) — Policy iteration
Run PI on the same MDP and confirm $V^{PI} = V^{VI}$ to FP precision.

> **Hint**: use the helper `policy_iteration`. Both methods are GPI; same fixed point.

In [ ]:
def exercise_3_policy_iteration(rows=5, cols=5, gamma=0.95):
    # YOUR CODE HERE
    return S.exercise_3_policy_iteration(rows=rows, cols=cols, gamma=gamma)


In [ ]:
V_vi, _, _ = exercise_2_value_iteration(gamma=0.95)
V_pi, pi, n = exercise_3_policy_iteration(gamma=0.95)
np.testing.assert_allclose(V_vi, V_pi, atol=1e-5)
print(f'Exercise 3: OK; PI converged in {n} outer sweeps')

## Exercise 4 (★★) — Stochastic GridWorld
Sweep slip probability $\rho\in\{0, 0.1, 0.2, 0.3, 0.4\}$ on a 4x4 grid with a pit at (1,1) and a goal at (3,3). Plot $V^\star(s_0)$ vs $\rho$.

> **Hint**: pass `slip_prob` to `GridWorld`. Recall: slipping introduces a uniform-random alternate action with probability $\rho$.

In [ ]:
def exercise_4_slip_world_optimal_value(slip=0.2, gamma=0.95):
    # YOUR CODE HERE
    return S.exercise_4_slip_world_optimal_value(slip=slip, gamma=gamma)


In [ ]:
slips = np.linspace(0, 0.4, 5)
vals = [exercise_4_slip_world_optimal_value(slip=float(s)) for s in slips]
fig, ax = plt.subplots(figsize=(5,3))
ax.plot(slips, vals, 'o-', color=RL_PALETTE['blue'])
ax.set_xlabel('slip probability'); ax.set_ylabel('$V^\\star(s_0)$');
ax.set_title('Optimal start-state value vs slip');
plt.show()
assert vals[0] > vals[-1]
print('Exercise 4: OK')

## Exercise 5 (★★) — Sup-norm contraction
Verify $\|T_\star V_1 - T_\star V_2\|_\infty \le \gamma \|V_1 - V_2\|_\infty$ for random $V$.

> **Hint**: this is Theorem 3.2 of the notes. Verifying it empirically is the easiest way to internalise it.

In [ ]:
def exercise_5_contraction_factor(env, V1, V2, gamma=0.95):
    # YOUR CODE HERE
    return S.exercise_5_contraction_factor(env, V1, V2, gamma=gamma)


In [ ]:
env = GridWorld(rows=4, cols=4, start=(0,0), goals=((3,3),))
rng = np.random.default_rng(0)
for _ in range(20):
    V1 = rng.standard_normal(env.nS); V2 = rng.standard_normal(env.nS)
    tdiff, diff = exercise_5_contraction_factor(env, V1, V2, gamma=0.9)
    assert tdiff <= 0.9 * diff + 1e-9, (tdiff, diff)
print('Exercise 5: contraction verified on 20 random pairs')

## Exercise 6 (★★) — Policy evaluation as a linear system
Implement closed-form policy evaluation via $V = (I-\gamma P^\pi)^{-1} r^\pi$.

> **Hint**: build $P^\pi$ as an $nS\times nS$ matrix and $r^\pi$ as a length-$nS$ vector by iterating over the transition table for the actions chosen by $\pi$.

In [ ]:
def exercise_6_policy_evaluation_closed_form(env, policy, gamma=0.9):
    # YOUR CODE HERE
    return S.exercise_6_policy_evaluation_closed_form(env, policy, gamma=gamma)


In [ ]:
env = GridWorld(rows=4, cols=4, start=(0,0), goals=((3,3),))
pi = np.array([1,1,1,2, 1,1,1,2, 1,1,1,2, 1,1,1,0])
V_lin = exercise_6_policy_evaluation_closed_form(env, pi, gamma=0.9)
V_iter = policy_evaluation(env, pi, gamma=0.9, tol=1e-12)
np.testing.assert_allclose(V_lin, V_iter, atol=1e-6)
print('Exercise 6: OK — V_linear matches V_iterative')

## Exercise 7 (★★★, open-ended) — Modified policy iteration
Implement GPI with `eval_sweeps` inner Bellman-expectation sweeps. Compare convergence (in outer iterations) for `eval_sweeps in [1, 3, 10, 50]`.

> **Hint**: `eval_sweeps=1` recovers VI; `eval_sweeps=large` recovers PI. Plot it and discuss the trade-off.

In [ ]:
def exercise_7_gpi(env, gamma=0.95, eval_sweeps=1, max_outer=50):
    # YOUR CODE HERE
    return S.exercise_7_gpi(env, gamma=gamma, eval_sweeps=eval_sweeps, max_outer=max_outer)


In [ ]:
env = GridWorld(rows=5, cols=5, start=(0,0), goals=((4,4),))
for k in (1, 3, 10, 50):
    V, pi, n = exercise_7_gpi(env, gamma=0.95, eval_sweeps=k, max_outer=200)
    print(f'  eval_sweeps={k:>3d}  outer iters={n}')
print('Exercise 7: OK')

## Exercise 8 (★★★, applied — matching) — Jack's car rental
Run value iteration on a `CarRentalMDP(cap=5, gamma=0.9)`. Plot the optimal policy as a heatmap (rows = location-1 stock, cols = location-2 stock, cell colour = move action).

> **Hint**: `pi[s_idx]` is the *index* into `mdp.actions_list`, not the move itself. Convert via `mdp.actions_list[pi[s_idx]]`.

In [ ]:
def exercise_8_car_rental(cap=5, gamma=0.9):
    # YOUR CODE HERE
    return S.exercise_8_car_rental(cap=cap, gamma=gamma)


In [ ]:
V, pi, n_iters = exercise_8_car_rental(cap=5, gamma=0.9)
from project.carrental import CarRentalMDP
mdp = CarRentalMDP(cap=5, gamma=0.9, poisson_cap=8)
policy_grid = np.zeros((mdp.cap+1, mdp.cap+1), dtype=int)
for s_idx, ai in enumerate(pi):
    n1, n2 = mdp.idx2s(s_idx)
    policy_grid[n1, n2] = mdp.actions_list[int(ai)]
fig, ax = plt.subplots(figsize=(4,4))
im = ax.imshow(policy_grid, cmap='RdBu', origin='lower')
ax.set_xlabel('cars at loc 2'); ax.set_ylabel('cars at loc 1')
ax.set_title("Jack's car rental — optimal #cars moved 1→2")
plt.colorbar(im, ax=ax)
plt.show()
print(f'Exercise 8: OK; converged in {n_iters} VI sweeps')

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

_A consolidated cheat-sheet — read only after you've attempted each exercise. Stronger than the inline hints above the scaffolds, weaker than the full solution. Worked answers are in `project/solutions.py` in the same folder (`week-02-bellman-dp`).
_

### Exercise 1 (★) — One step of the Bellman optimality operator

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 2 (★) — Value iteration

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 3 (★★) — Policy iteration

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 4 (★★) — Stochastic GridWorld

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 5 (★★) — Sup-norm contraction

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 6 (★★) — Policy evaluation as a linear system

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 7 (★★★, open-ended) — Modified policy iteration

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 8 (★★★, applied — matching) — Jack's car rental

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.


---
_End of week. Mark this notebook in your tracker only after every exercise's hidden test cell passes._
